In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib
import os

print("Modélisation - Bibliothèques chargées")

In [ ]:
df = pd.read_csv("Projet_Data/processed/iot_preprocessed.csv")

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

print(f"Dimensions du dataset : {df.shape}")

In [ ]:
targets = ['target_temp', 'target_humidity', 'target_co2']
excluded_columns = ["time", "deveui"] + targets

features = [col for col in df.columns if col not in excluded_columns]

print(f"Nombre de variables explicatives : {len(features)}")
print(f"Variables cibles : {targets}")

X = df[features]
y = df[targets]

In [ ]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Entraînement du modèle en cours (cela peut prendre un moment)...")
rf.fit(X_train, y_train)

print("Modèle entraîné avec succès !")

In [ ]:
# Sauvegarde du modèle pour une utilisation future
os.makedirs("Projet_Data/models", exist_ok=True)
model_path = "Projet_Data/models/rf_multi_output.joblib"

joblib.dump(rf, model_path)
print(f"\nModèle sauvegardé avec succès sous : {model_path}")

In [ ]:
import matplotlib.pyplot as plt

# Génération des prédictions sur le jeu de test
y_pred_rf = rf.predict(X_test)

plt.figure(figsize=(15, 5))
# On affiche les 100 premières prédictions de Température pour plus de lisibilité
plt.plot(y_test['target_temp'].values[:100], label='Température Réelle', alpha=0.8)
plt.plot(y_pred_rf[:, 0][:100], label='Prédiction Random Forest', linestyle='--', color='red')
plt.title("Échec du Random Forest : Incapacité à lisser la tendance temporelle")
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

print("Entraînement Linear Regression...")
lr = LinearRegression()
lr.fit(X_train, y_train)
print(f"LR RMSE global: {np.sqrt(mean_squared_error(y_test, lr.predict(X_test))):.2f}")

print("Entraînement SVR...")
svr = MultiOutputRegressor(LinearSVR(random_state=42, max_iter=1000))
svr.fit(X_train, y_train)
print(f"SVR RMSE global: {np.sqrt(mean_squared_error(y_test, svr.predict(X_test))):.2f}")

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("Projet_Data/images", exist_ok=True)

models = ['LSTM', 'GRU', 'CNN', 'Transformer']
rmse_temp = [1.441, 1.450, 1.456, 1.476]
rmse_hum = [3.320, 3.284, 3.307, 3.353]
rmse_co2 = [3782.84, 3784.05, 3784.50, 3783.13]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

axes[0].bar(models, rmse_temp, color=colors, edgecolor='black')
axes[0].set_title('RMSE - Température (°C)', fontweight='bold')
axes[0].set_ylim(1.4, 1.5)

axes[1].bar(models, rmse_hum, color=colors, edgecolor='black')
axes[1].set_title('RMSE - Humidité (%)', fontweight='bold')
axes[1].set_ylim(3.2, 3.4)

axes[2].bar(models, rmse_co2, color=colors, edgecolor='black')
axes[2].set_title('RMSE - CO₂ (ppm)', fontweight='bold')
axes[2].set_ylim(3780, 3786)

for ax in axes:
    ax.tick_params(axis='x', rotation=15)

plt.suptitle("Figure 3.11 : Comparaison des erreurs RMSE des modèles Deep Learning", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("Projet_Data/images/figure_3_11.png", bbox_inches='tight')
print("Graphique sauvegardé sous : Projet_Data/images/figure_3_11.png")
plt.show()